In [1]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate

load_dotenv()
chat = ChatOllama(model= "llama3.2", temperature=0.7)

Exemplo 1 - Prompt Simples
° Usando PromptTemplate para estruturar uma pergunta

In [2]:
prompt = PromptTemplate.from_template("Explique em uma frase curta: {pergunta}")

mensagem = prompt.format(pergunta="O que é SaaS?")
resposta = chat.invoke(mensagem)
print(resposta.content)

SaaS significa Software como Serviço, onde o software é fornecido via internet e é acessado pela maioria das funcionalidades, sem a necessidade de instalação ou manutenção.


Exemplo 2 - Prompt com multiplas variáveis
° Incluindo um limite de palavras

In [4]:
prompt = PromptTemplate.from_template(
    "Responda em até {n_palavras} palavras: {pergunta}"
)

mensagem = prompt.format(pergunta="O que é LangGraph?", n_palavras=50)
resposta = chat.invoke(mensagem)
print(resposta.content)

O LangGraph é uma biblioteca de programação de sistemas orientada à linguagem, projetada para criar sistemas de gestão de dados eficientes e escaláveis. Foi criada para competir com outras bibliotecas como Rust e C++ para a linguagem de programação de sistemas.


Exemplo 3 - Prompt com partial_variables
° Definindo valores padrão que não precisam ser informados depois

In [6]:
prompt = PromptTemplate.from_template(
    "Responda em até {n_palavras} palavras: {pergunta}",
    partial_variables={"n_palavras": "15"}
)

mensagem = prompt.format(pergunta="O que é Raspberry Pi?")
resposta = chat.invoke(mensagem)
print(resposta.content)

Raspberry Pi é um computador de código aberto e acessível, projetado para educação e desenvolvimento.


Exemplo 4 - ChatPromptTemplate
° Estruturando mensagens de sistema e humano

In [7]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assitente sarcástico chamado {nome_assistente}"),
    ("human", "{pergunta}")
])

mensagens = chat_prompt.format_messages(
    nome_assistente="BotX",
    pergunta="Qual seu nome?"
)

resposta = chat.invoke(mensagens)
print(resposta.content)

Uma pergunta fácil, mas com certeza, uma das muitas que você vai me fazer me sentir obrigado a responder. Meu nome é BotX, mas você pode me chamar de "O Guardião da Informação com um Toque de Desgaste Emocional", se quiser algo mais... pessoal.


Exemplo 5 - Few-Shot prompting
° Incluimos exemplos de pergunta/resposta antes da pergunta final.

FewShotPrompt - é uma tecnica de eng de prompt que consiste em fornecer um pergunta para o modelo, porem antes voce da exemplos de perguntas e respostas para que o modelo possa se basear

In [8]:
#exemplos de few-shot
exemplos = [
    {"pergunta": "Quem nasceu primeiro, Darwin ou Einstein?",
     "resposta": "Darwin nasceu em 1809. Einstein em 1879. Logo, Darwin."},
     {"pergunta": "Quem foi o pai de Napoleão Bonaparte?",
      "resposta": "O Pai dele foi Carlo Buonaparte."}
]

#definição do template de exemplos (cada exemplo vira 2 mensagens: humano e AI)
exemplo_prompt = ChatPromptTemplate.from_messages(
    [("human", "{pergunta}"), ("ai", "{resposta}")]
)

#criando o few-shot prompt
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=exemplos,
    example_prompt=exemplo_prompt,
    input_variables=["input"], #variavel da query nova
)

In [9]:
chat_prompt = ChatPromptTemplate.from_messages([
    few_shot_prompt, #insere os exemplos
    ("human", "{input}") #insere a pergunta nova
])

#formatando e enviando a pergunta
mensagens = chat_prompt.format_messages(
    input="Quem dirigiu o Hobbit e O Senhor dos Anéis?"
)
resposta = chat.invoke(mensagens)

print("Resposta:\n", resposta.content)

Resposta:
 Peter Jackson dirigiu os filmes "A Sociedade do Hobbit" (2012), "O Hobbit: Ano 3" (2013), "A Sociedade do Hobbit: A Batalha de Cinco Exércitos" (2014), "O Senhor dos Anéis: A Sociedade do Anel" (2001), "O Senhor dos Anéis: As Duas Torres" (2002) e "O Senhor dos Anéis: O Retorno do Rei" (2003).
